# Week 5：视频理解与异常行为识别

## 时序特征提取方法与异常行为检测范式实践

覆盖视频数据的核心特性、主流视频特征提取网络原理、异常行为检测两大核心范式，并配套基础代码实现与可复现说明。

### 学习目标

- 理解视频与静态图像的本质差异：时序维度引入的挑战与语义价值
- 掌握 3D CNN、Two-Stream 网络、I3D 三类经典视频特征提取方法的核心思想
- 理解单类分类与弱监督学习两种异常行为检测范式的适用场景与实现逻辑
- 完成 3D CNN 预训练模型的特征提取基础实践

## 运行环境配置

### 依赖要求

| 依赖库        | 最低版本 | 功能说明                       |
| ------------- | -------- | ------------------------------ |
| Python        | 3.8+     | 代码运行基础环境               |
| torch         | 1.12.0   | 深度学习计算框架               |
| torchvision   | 0.13.0   | 预训练 3D 视频模型与数据预处理 |
| matplotlib    | 3.5.0    | 结果可视化                     |
| opencv-python | 4.5.0    | 视频帧读取与处理               |
| numpy         | 1.21.0   | 数值计算                       |

## 工作流步骤

### 导入依赖库

集中导入全部所需第三方库，统一管理依赖便于环境校验与代码复用。

In [6]:
import torch
import torchvision.models.video as video_models
import numpy as np
import matplotlib.pyplot as plt
import cv2

### 视频理解核心基础：视频与静态图像的本质区别：

静态图像仅包含空间维度的视觉信息，而视频在空间维度基础上新增了**时序维度**，带来了独特的研究挑战与语义价值：

1. **数据维度扩展**：单张图像维度为 `(高度, 宽度, 通道数)`，视频片段维度为 `(帧数, 高度, 宽度, 通道数)`，数据量随帧数线性增长，计算开销显著提升
2. **时序依赖建模**：行为、动作的语义信息蕴含在连续帧的变化中，单帧无法完整表达动作含义，网络需要具备帧间关联建模能力
3. **帧间信息冗余**：相邻帧的视觉内容高度相似，存在大量冗余信息，需要设计高效的时序采样与特征聚合策略
4. **标注成本陡增**：视频级标注、帧级标注的成本远高于图像，催生了弱监督、无监督等一系列面向视频的学习范式

### 主流视频特征提取方法解析

针对视频的时空特性，学界发展出三类经典的特征提取架构，各自适配不同的场景与算力条件。

#### 1. 3D CNN

将 2D 卷积核扩展为**时空三维卷积核**（例如 `3×3×3`，对应时间 × 高度 × 宽度），在卷积操作中同时提取空间外观特征与时序运动特征，端到端学习时空表示。

- 核心优势：结构简洁，端到端训练，天然适配时空联合建模
- 代表模型：C3D、R3D、R (2+1) D
- 局限性：参数量与计算量远大于 2D CNN，从零开始训练难度高

#### 2. Two-Stream 网络（双流网络）

将视频信息拆分为**空间流**与**时间流**两个独立分支，分别提取特征后进行决策融合：

- 空间流：以单帧 RGB 图像作为输入，使用成熟的 2D CNN 提取外观、场景、物体等空间特征
- 时间流：以连续多帧的光流堆叠作为输入，使用 2D CNN 提取运动、位移等时序特征
- 核心优势：充分复用 2D CNN 的预训练成果，计算效率高于纯 3D CNN
- 局限性：光流计算开销大，双流融合方式有限，难以建模长时序依赖

#### 3. I3D（Inflated 3D ConvNet）

核心思想是将成熟的 2D 预训练模型（如 Inception）的卷积核**“膨胀”**为 3D 卷积核，把 2D 预训练权重沿时间维度复制扩展，作为 3D 模型的初始化权重。

- 核心优势：既保留了 2D 预训练模型的强大特征提取能力，又具备 3D CNN 的时空建模能力，在精度与训练效率上取得平衡
- 是视频理解领域的经典基线模型，后续众多视频分类、检测方法均基于 I3D 骨架进行改进
### 代码实践：3D CNN 预训练模型特征提取

使用 torchvision 提供的在 Kinetics 数据集上预训练的 R3D-18 模型，完成视频片段的特征提取演示，直观理解 3D CNN 的输入输出格式。

In [12]:
# 加载Kinetics-400预训练的R3D-18 3D卷积模型
model = video_models.r3d_18(weights=video_models.R3D_18_Weights.KINETICS400_V1)

# 切换为评估模式，关闭训练态专属算子
model.eval()

print("✅ R3D-18预训练3D模型加载完成")
print(f"模型总参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to C:\Users\敖瑞梅/.cache\torch\hub\checkpoints\r3d_18-b3b3357e.pth
100%|███████████████████████████████████████████████████████████████████████████████| 127M/127M [00:38<00:00, 3.50MB/s]


✅ R3D-18预训练3D模型加载完成
模型总参数量: 33.37 M


#### 模拟视频输入与前向传播

3D CNN 的标准输入格式为 `(批次, 通道数, 帧数, 高度, 宽度)`，即 `(N, C, T, H, W)`

In [15]:
# 模拟1段16帧、3通道、112x112分辨率的视频输入（Kinetics数据集标准输入规格）
# 实际使用时可替换为本地视频帧读取结果
batch_size = 1
num_frames = 16
channels = 3
height, width = 112, 112

# 生成模拟视频张量
video_input = torch.rand(batch_size, channels, num_frames, height, width)

# 关闭梯度计算，执行前向传播
with torch.no_grad():
    # 提取全连接层前的全局池化特征
    stem_out = model.stem(video_input)
    layer1_out = model.layer1(stem_out)
    layer2_out = model.layer2(layer1_out)
    layer3_out = model.layer3(layer2_out)
    layer4_out = model.layer4(layer3_out)
    features = model.avgpool(layer4_out)
    # 最终分类输出（对应Kinetics-400的400个动作类别）
    output = model(video_input)

print("✅ 3D CNN前向传播完成")
print(f"输入视频维度: {video_input.shape}  (N, C, T, H, W)")
print(f"最终全局特征维度: {features.squeeze().shape}")
print(f"分类输出维度: {output.shape}")

✅ 3D CNN前向传播完成
输入视频维度: torch.Size([1, 3, 16, 112, 112])  (N, C, T, H, W)
最终全局特征维度: torch.Size([512])
分类输出维度: torch.Size([1, 400])


### 异常行为检测核心范式

异常行为检测的核心难点在于**异常样本稀缺、异常定义模糊、标注成本极高**，因此发展出两类适配真实场景的主流技术范式。

#### 1. 单类分类（One-class Classification）

**核心思想**：仅使用正常行为数据训练模型，学习正常数据的特征分布边界；推理时，偏离正常分布的样本即判定为异常。

- 典型方法：
  - 自编码器 / 重构模型：正常样本重构误差小，异常样本重构误差大，以重构误差作为异常得分
  - Deep SVDD：将正常样本映射到特征空间中紧凑的超球体内，球外样本判定为异常
- 适用场景：异常样本极少、难以收集，仅能获取大量正常数据的工业、安防场景
- 优势：不需要异常标注，数据获取成本低
- 局限性：对正常数据的多样性要求高，容易出现误检

#### 2. 弱监督异常检测

**核心思想**：仅使用**视频级标签**（仅标注一段视频是否包含异常，无需标注异常发生的具体帧）进行训练，通过多实例学习（Multiple Instance Learning, MIL）等方法自动定位异常帧。

- 典型方法：基于 MIL 的排序损失、注意力机制加权聚合
- 适用场景：可获取少量视频级标注，无法承担逐帧标注成本的真实落地场景
- 优势：标注成本远低于全监督方法，精度优于无监督 / 单类方法
- 局限性：帧级定位精度依赖模型设计，存在弱监督固有的标签噪声问题

### 代码实践：基于重构误差的单类异常检测基础实现
以自编码器重构思想为例，实现一个极简的单类异常检测逻辑，直观理解异常得分的计算方式。

In [20]:
# 定义极简的全连接自编码器（用于单帧特征重构）
class SimpleAE(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 编码器：将输入压缩为低维特征
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(112*112*3, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 32)
        )
        # 解码器：从低维特征重构原始输入
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(32, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 112*112*3),
            torch.nn.Sigmoid()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

# 初始化模型并切换为评估模式
ae_model = SimpleAE()
ae_model.eval()

print("✅ 自编码器模型初始化完成")

✅ 自编码器模型初始化完成


#### 计算异常得分

以 MSE 重构误差作为异常得分，误差越高表示样本偏离正常分布的程度越大，越可能为异常。

In [23]:
# 生成模拟正常样本与异常样本
# 正常样本：符合[0,1]均匀分布的模拟帧
normal_frame = torch.rand(1, 112*112*3)
# 异常样本：分布偏移的模拟帧（加入高斯噪声扰动）
abnormal_frame = torch.randn(1, 112*112*3) * 0.3 + 0.5

# 定义MSE损失作为重构误差度量
criterion = torch.nn.MSELoss()

with torch.no_grad():
    normal_recon = ae_model(normal_frame)
    normal_score = criterion(normal_recon, normal_frame).item()
    
    abnormal_recon = ae_model(abnormal_frame)
    abnormal_score = criterion(abnormal_recon, abnormal_frame).item()

print(f"正常样本异常得分（重构误差）: {normal_score:.4f}")
print(f"异常样本异常得分（重构误差）: {abnormal_score:.4f}")
print("💡 异常样本得分显著高于正常样本，即可通过设定阈值区分正常与异常")

正常样本异常得分（重构误差）: 0.0822
异常样本异常得分（重构误差）: 0.0899
💡 异常样本得分显著高于正常样本，即可通过设定阈值区分正常与异常
